In [21]:
import pandas as pd # note bookul asta e slightly vibecoded cred (l-am primit de la Circi si am schimbat eu cateva lucruri :)))

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

train_df.head()

,IDSample,FrameNumber,J1X,J1Y,J1Z,J2X,J2Y,J2Z,J3X,J3Y,...,J23Y,J23Z,J24X,J24Y,J24Z,J25X,J25Y,J25Z,Action,Camera
0,1,1,0.164490,0.291355,3.091319,0.189296,0.615010,3.080443,0.209298,0.928650,...,0.249604,2.929706,0.237679,0.185178,2.881282,0.290780,0.236038,2.852048,3,1
1,1,2,0.164051,0.291135,3.091265,0.187257,0.614560,3.080929,0.206220,0.927991,...,0.242949,2.934857,0.237713,0.184868,2.881657,0.288819,0.236270,2.851250,3,1
2,1,3,0.160394,0.291922,3.091712,0.183053,0.614688,3.081785,0.202935,0.927368,...,0.261962,2.940350,0.237603,0.182165,2.880324,0.286495,0.221468,2.868750,3,1
3,1,4,0.157784,0.291979,3.090791,0.179416,0.614119,3.081133,0.198786,0.925982,...,0.257649,2.934111,0.231934,0.184934,2.879659,0.289556,0.229324,2.863987,3,1
4,1,5,0.154023,0.289773,3.091381,0.174946,0.611712,3.081385,0.194107,0.923457,...,0.252397,2.932880,0.223781,0.178191,2.879539,0.278711,0.216850,2.864786,3,1


In [4]:
test_frame_counts = test_df.groupby('IDSample').size().reset_index(name='frame_count')

subtask1_output = []
for idx, row in test_frame_counts.iterrows():
    subtask1_output.append({
        'subtaskID': 1,
        'datapointID': int(row['IDSample']),
        'answer': int(row['frame_count'])
    })

subtask1_output[0]

{'subtaskID': 1, 'datapointID': 3361, 'answer': 124}

In [5]:
def extract_features_simple(df):
    joint_cols = [col for col in df.columns if col.startswith('J')]

    features = df.groupby('IDSample')[joint_cols].agg([
        'mean', 'std', 'min', 'max',
        lambda x: x.quantile(0.25),  # Q1
        lambda x: x.quantile(0.75),  # Q3
    ])

    features.columns = ['_'.join(col).strip() for col in features.columns.values]
    frame_counts = df.groupby('IDSample').size()
    features['num_frames'] = frame_counts

    return features

train_features_df = extract_features_simple(train_df)
test_features_df = extract_features_simple(test_df)

train_labels = train_df.groupby('IDSample')[['Action', 'Camera']].first()
y_action = train_labels['Action'].values
y_camera = train_labels['Camera'].values

X_train = train_features_df.values
X_test = test_features_df.values

X_train.shape

(3354, 451)

In [8]:
import numpy as np
from sklearn.preprocessing import StandardScaler

X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train[0][0]

np.float64(0.11652087859649123)

In [12]:
from lightgbm import LGBMClassifier

action_model = LGBMClassifier(random_state=42, verbose=-1)
action_model.fit(X_train_scaled, y_action)

train_score = action_model.score(X_train_scaled, y_action)
train_score

C:\Users\Ilie-Goga\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


1.0

In [15]:
action_predictions = action_model.predict(X_test_scaled)

subtask2_output = []
test_sample_ids = test_features_df.index.values
for sample_id, pred in zip(test_sample_ids, action_predictions):
    subtask2_output.append({
        'subtaskID': 2,
        'datapointID': int(sample_id),
        'answer': int(pred)
    })

subtask2_output[0]

C:\Users\Ilie-Goga\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


{'subtaskID': 2, 'datapointID': 3361, 'answer': 2}

In [16]:
camera_model = LGBMClassifier(random_state=42, verbose=-1)
camera_model.fit(X_train_scaled, y_camera)

train_score = camera_model.score(X_train_scaled, y_camera)
train_score

C:\Users\Ilie-Goga\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


1.0

In [18]:
camera_predictions = camera_model.predict(X_test_scaled)

subtask3_output = []
for sample_id, pred in zip(test_sample_ids, camera_predictions):
    subtask3_output.append({
        'subtaskID': 3,
        'datapointID': int(sample_id),
        'answer': int(pred)
    })

subtask3_output[0]

C:\Users\Ilie-Goga\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


{'subtaskID': 3, 'datapointID': 3361, 'answer': 2}

In [19]:
all_outputs = subtask1_output + subtask2_output + subtask3_output
output_df = pd.DataFrame(all_outputs)
output_df.head()

,subtaskID,datapointID,answer
0,1,3361,124
1,1,3362,78
2,1,3363,207
3,1,3364,66
4,1,3365,83


In [20]:
output_df.to_csv('output.csv', index=False)